# 01a Ingest OpenSky History

Extract bounded historical Frankfurt slices from OpenSky Trino and land them into Bronze tables.

This notebook is the **historical backbone** of the project. It does not clean data and it does not compute complexity metrics. Its only responsibility is controlled historical ingestion.

## Target Tables

- `adsb_airspace_eddf.brz_adsb.hist_state_vectors`
- `adsb_airspace_eddf.brz_adsb.hist_flights`
- `adsb_airspace_eddf.obs.ingestion_log`
- `adsb_airspace_eddf.obs.ingestion_partition_log`

## Partition Strategy

- `state_vectors_data4` is processed by `hour`
- `flights_data4` is processed by `day`

## `dry_run` Semantics

When `dry_run=true`, the notebook:

- reads configs and widgets
- builds the planned hour/day partitions
- validates SQL templates
- optionally tests source connectivity with lightweight queries
- does **not** write Bronze tables
- does **not** overwrite partitions
- does **not** write to `obs` tables

Recommended partition statuses for the later implementation:

- `success`
- `failed`
- `empty_source`
- `skipped`
- `dry_run`

## Validation Tip

For early smoke tests, set `start_hour_utc` and `end_hour_utc` to the same UTC hour so the notebook extracts only one `hour` partition.

In [ ]:
from __future__ import annotations

from datetime import date, datetime, time, timedelta, timezone
from pathlib import Path
import json
import sys
import uuid
import yaml

if "spark" not in globals():
    raise RuntimeError("This notebook expects a Spark session, for example inside Databricks.")

UTC = timezone.utc
repo_root = Path.cwd().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

with (repo_root / "configs" / "region_config.yaml").open("r", encoding="utf-8") as handle:
    region_config = yaml.safe_load(handle)

with (repo_root / "configs" / "pipeline_config.yaml").open("r", encoding="utf-8") as handle:
    pipeline_config = yaml.safe_load(handle)

def ensure_iso_date_string(value) -> str:
    if hasattr(value, "isoformat"):
        return value.isoformat()
    return str(value)

default_catalog = pipeline_config.get("catalog_name", "adsb_airspace_eddf")
trino_defaults = pipeline_config.get("trino_connection", {})
default_start_date = ensure_iso_date_string(pipeline_config["study_window"]["start_date"])
default_end_date = ensure_iso_date_string(pipeline_config["study_window"]["end_date"])
default_start_hour_utc = ""
default_end_hour_utc = ""
default_trino_host = str(trino_defaults.get("host", "trino.opensky-network.org"))
default_trino_port = str(trino_defaults.get("port", 443))
default_trino_catalog = str(trino_defaults.get("catalog", "minio"))
default_trino_schema = str(trino_defaults.get("schema", "osky"))
default_trino_secret_scope = str(trino_defaults.get("secret_scope", "opensky"))
default_trino_username_key = str(trino_defaults.get("username_key", "username"))
default_trino_password_key = str(trino_defaults.get("password_key", "password"))

catalog = default_catalog
start_date_raw = default_start_date
end_date_raw = default_end_date
start_hour_utc_raw = default_start_hour_utc
end_hour_utc_raw = default_end_hour_utc
include_flights_raw = "true"
dry_run_raw = "true"
state_table = "state_vectors_data4"
flight_table = "flights_data4"
overwrite_empty_partitions_raw = "false"
trino_host_raw = default_trino_host
trino_port_raw = default_trino_port
trino_catalog_raw = default_trino_catalog
trino_schema_raw = default_trino_schema
trino_user_raw = ""
trino_password_raw = ""
trino_secret_scope_raw = default_trino_secret_scope
trino_secret_username_key_raw = default_trino_username_key
trino_secret_password_key_raw = default_trino_password_key

if "dbutils" in globals():
    dbutils.widgets.text("catalog", default_catalog)
    dbutils.widgets.text("start_date", default_start_date)
    dbutils.widgets.text("end_date", default_end_date)
    dbutils.widgets.text("start_hour_utc", default_start_hour_utc)
    dbutils.widgets.text("end_hour_utc", default_end_hour_utc)
    dbutils.widgets.dropdown("include_flights", "true", ["true", "false"])
    dbutils.widgets.dropdown("dry_run", "true", ["true", "false"])
    dbutils.widgets.text("state_table", state_table)
    dbutils.widgets.text("flight_table", flight_table)
    dbutils.widgets.dropdown("overwrite_empty_partitions", "false", ["true", "false"])
    dbutils.widgets.text("trino_host", default_trino_host)
    dbutils.widgets.text("trino_port", default_trino_port)
    dbutils.widgets.text("trino_catalog", default_trino_catalog)
    dbutils.widgets.text("trino_schema", default_trino_schema)
    dbutils.widgets.text("trino_user", "")
    dbutils.widgets.text("trino_password", "")
    dbutils.widgets.text("trino_secret_scope", default_trino_secret_scope)
    dbutils.widgets.text("trino_secret_username_key", default_trino_username_key)
    dbutils.widgets.text("trino_secret_password_key", default_trino_password_key)

    catalog = dbutils.widgets.get("catalog").strip() or default_catalog
    start_date_raw = dbutils.widgets.get("start_date").strip() or default_start_date
    end_date_raw = dbutils.widgets.get("end_date").strip() or default_end_date
    start_hour_utc_raw = dbutils.widgets.get("start_hour_utc").strip()
    end_hour_utc_raw = dbutils.widgets.get("end_hour_utc").strip()
    include_flights_raw = dbutils.widgets.get("include_flights")
    dry_run_raw = dbutils.widgets.get("dry_run")
    state_table = dbutils.widgets.get("state_table").strip() or state_table
    flight_table = dbutils.widgets.get("flight_table").strip() or flight_table
    overwrite_empty_partitions_raw = dbutils.widgets.get("overwrite_empty_partitions")
    trino_host_raw = dbutils.widgets.get("trino_host").strip() or default_trino_host
    trino_port_raw = dbutils.widgets.get("trino_port").strip() or default_trino_port
    trino_catalog_raw = dbutils.widgets.get("trino_catalog").strip() or default_trino_catalog
    trino_schema_raw = dbutils.widgets.get("trino_schema").strip() or default_trino_schema
    trino_user_raw = dbutils.widgets.get("trino_user").strip()
    trino_password_raw = dbutils.widgets.get("trino_password")
    trino_secret_scope_raw = dbutils.widgets.get("trino_secret_scope").strip() or default_trino_secret_scope
    trino_secret_username_key_raw = dbutils.widgets.get("trino_secret_username_key").strip() or default_trino_username_key
    trino_secret_password_key_raw = dbutils.widgets.get("trino_secret_password_key").strip() or default_trino_password_key

focus_airport = region_config["focus_airport"]
bbox = region_config["regional_bbox"]
scope_id = "fra_case_study_v1"

def parse_bool(raw_value: str) -> bool:
    return raw_value.strip().lower() == "true"

def parse_iso_date(raw_value: str) -> date:
    return datetime.strptime(raw_value, "%Y-%m-%d").date()

def parse_int(raw_value: str) -> int:
    return int(raw_value.strip())

def parse_optional_hour(raw_value: str) -> int | None:
    text = raw_value.strip()
    if not text:
        return None
    value = int(text)
    if value < 0 or value > 23:
        raise ValueError("Hour widgets must be between 0 and 23 inclusive")
    return value

include_flights = parse_bool(include_flights_raw)
dry_run = parse_bool(dry_run_raw)
overwrite_empty_partitions = parse_bool(overwrite_empty_partitions_raw)
trino_host = trino_host_raw.strip() or default_trino_host
trino_port = parse_int(trino_port_raw or default_trino_port)
trino_catalog = trino_catalog_raw.strip() or default_trino_catalog
trino_schema = trino_schema_raw.strip() or default_trino_schema
trino_user = trino_user_raw.strip()
trino_password = trino_password_raw
trino_secret_scope = trino_secret_scope_raw.strip()
trino_secret_username_key = trino_secret_username_key_raw.strip()
trino_secret_password_key = trino_secret_password_key_raw.strip()

start_date = parse_iso_date(start_date_raw)
end_date = parse_iso_date(end_date_raw)
start_hour_utc = parse_optional_hour(start_hour_utc_raw)
end_hour_utc = parse_optional_hour(end_hour_utc_raw)
if start_date > end_date:
    raise ValueError("start_date cannot be after end_date")
if (start_hour_utc is None) != (end_hour_utc is None):
    raise ValueError("Specify both start_hour_utc and end_hour_utc, or leave both blank")
if start_hour_utc is not None and start_date == end_date and start_hour_utc > end_hour_utc:
    raise ValueError("When start_date=end_date, start_hour_utc cannot be after end_hour_utc")

run_id = f"history_ingest_{datetime.now(UTC).strftime('%Y%m%dT%H%M%SZ')}_{uuid.uuid4().hex[:8]}"


In [ ]:
def hourly_partitions(start_dt: datetime, end_dt: datetime) -> list[datetime]:
    values = []
    current = start_dt
    while current <= end_dt:
        values.append(current)
        current += timedelta(hours=1)
    return values

def daily_partitions(start_d: date, end_d: date) -> list[date]:
    values = []
    current = start_d
    while current <= end_d:
        values.append(current)
        current += timedelta(days=1)
    return values

start_time_utc = time(hour=start_hour_utc) if start_hour_utc is not None else time.min
end_time_utc = time(hour=end_hour_utc, minute=59, second=59) if end_hour_utc is not None else time.max.replace(microsecond=0)

start_dt = datetime.combine(start_date, start_time_utc, tzinfo=UTC)
end_dt = datetime.combine(end_date, end_time_utc, tzinfo=UTC)

planned_hours = hourly_partitions(start_dt.replace(minute=0, second=0), end_dt.replace(minute=0, second=0))
planned_days = daily_partitions(start_date, end_date) if include_flights else []

def hour_epoch(dt: datetime) -> int:
    return int(dt.timestamp())

def day_epoch(d: date) -> int:
    return int(datetime.combine(d, time.min, tzinfo=UTC).timestamp())

def build_state_vectors_sql(hour_partition_epoch: int) -> str:
    return f"""
    SELECT
      time, icao24, callsign, lat, lon, velocity, heading, vertrate,
      onground, alert, spi, squawk, baroaltitude, geoaltitude,
      lastposupdate, lastcontact, hour
    FROM {state_table}
    WHERE hour = {hour_partition_epoch}
      AND lat BETWEEN {bbox['min_latitude']} AND {bbox['max_latitude']}
      AND lon BETWEEN {bbox['min_longitude']} AND {bbox['max_longitude']}
    """.strip()

def build_flights_sql(day_partition_epoch: int) -> str:
    return f"""
    SELECT
      icao24, firstseen, lastseen, estdepartureairport,
      estarrivalairport, callsign, day
    FROM {flight_table}
    WHERE day = {day_partition_epoch}
      AND (
        estdepartureairport = '{focus_airport}'
        OR estarrivalairport = '{focus_airport}'
      )
    """.strip()

planned_state_queries = [
    {"partition_key": "hour", "partition_value": hour_epoch(dt), "sql": build_state_vectors_sql(hour_epoch(dt))}
    for dt in planned_hours
]

planned_flight_queries = [
    {"partition_key": "day", "partition_value": day_epoch(d), "sql": build_flights_sql(day_epoch(d))}
    for d in planned_days
]


In [ ]:
run_plan = {
    "run_id": run_id,
    "catalog": catalog,
    "focus_airport": focus_airport,
    "source_objects": {
        "states": state_table,
        "flights": flight_table if include_flights else None,
    },
    "target_tables": {
        "states": f"{catalog}.brz_adsb.hist_state_vectors",
        "flights": f"{catalog}.brz_adsb.hist_flights",
        "run_log": f"{catalog}.obs.ingestion_log",
        "partition_log": f"{catalog}.obs.ingestion_partition_log",
    },
    "date_window": {
        "start_date": start_date.isoformat(),
        "end_date": end_date.isoformat(),
        "start_hour_utc": start_hour_utc,
        "end_hour_utc": end_hour_utc,
    },
    "dry_run": dry_run,
    "overwrite_empty_partitions": overwrite_empty_partitions,
    "planned_state_partition_count": len(planned_state_queries),
    "planned_flight_partition_count": len(planned_flight_queries),
    "first_state_query": planned_state_queries[0]["sql"] if planned_state_queries else None,
    "first_flight_query": planned_flight_queries[0]["sql"] if planned_flight_queries else None,
}

run_plan

## Execution Notes

This notebook now resolves credentials, builds partition plans, extracts OpenSky history, writes Bronze tables, and records run metadata.

- `src/ingestion/opensky_trino_client.py`

Recommended credential order:

- `trino_user` must be your lowercase OpenSky website username
- the Python client uses external OAuth2 authentication and should prompt a browser-based login flow
- Databricks secrets or widgets can still provide the username, but direct password entry is not used by the Trino client path

In [ ]:
import pandas as pd

def coalesce_text(*values) -> str:
    for value in values:
        if value is None:
            continue
        text = str(value).strip()
        if text:
            return text
    return ""

def try_get_secret(scope: str, key: str) -> str:
    if "dbutils" not in globals():
        return ""
    if not scope or not key:
        return ""
    try:
        return dbutils.secrets.get(scope=scope, key=key)
    except Exception:
        return ""

def utc_now() -> datetime:
    return datetime.now(UTC)

def naive_utc(value: datetime | None) -> datetime | None:
    if value is None:
        return None
    if value.tzinfo is None:
        return value
    return value.astimezone(UTC).replace(tzinfo=None)

def json_dumps(payload: dict) -> str:
    return json.dumps(payload, sort_keys=True, default=str)

def normalize_scalar(value):
    if value is None:
        return None
    if hasattr(value, "to_pydatetime"):
        return value.to_pydatetime()
    try:
        if pd.isna(value):
            return None
    except TypeError:
        pass
    return value

def create_dataframe_for_table(*, target_table: str, rows: list[dict]):
    ensure_target_table_exists(target_table)
    target_schema = spark.table(target_table).schema
    normalized_rows = [
        {
            field.name: normalize_scalar(row.get(field.name))
            for field in target_schema
        }
        for row in rows
    ]
    return spark.createDataFrame(normalized_rows, schema=target_schema)

def build_trino_client():
    secret_user = try_get_secret(trino_secret_scope, trino_secret_username_key)

    from src.ingestion.opensky_trino_client import OpenSkyTrinoClient, build_config

    config = build_config(
        user=coalesce_text(secret_user, trino_user),
        host=trino_host,
        port=trino_port,
        catalog=trino_catalog,
        schema=trino_schema,
    )
    return OpenSkyTrinoClient(config)

def ensure_target_table_exists(table_name: str) -> None:
    if not spark.catalog.tableExists(table_name):
        raise ValueError(f"Missing target table: {table_name}. Run 00_platform_setup_catalog_schema.ipynb first.")

def create_target_dataframe(*, pdf, target_table: str, query_started_at: datetime, query_completed_at: datetime):
    payload_pdf = pdf.copy()
    payload_pdf["source_query_start"] = naive_utc(query_started_at)
    payload_pdf["source_query_end"] = naive_utc(query_completed_at)
    payload_pdf["ingested_at"] = naive_utc(query_completed_at)
    payload_pdf["run_id"] = run_id

    target_schema = spark.table(target_table).schema
    target_columns = [field.name for field in target_schema]
    for column_name in target_columns:
        if column_name not in payload_pdf.columns:
            payload_pdf[column_name] = None
    payload_pdf = payload_pdf[target_columns]

    payload_pdf = payload_pdf.where(pd.notnull(payload_pdf), None)
    return create_dataframe_for_table(
        target_table=target_table,
        rows=payload_pdf.to_dict(orient="records"),
    )

def append_log_row(*, target_table: str, payload: dict) -> None:
    log_df = create_dataframe_for_table(target_table=target_table, rows=[payload])
    log_df.write.mode("append").saveAsTable(target_table)

def extract_partition_via_trino(*, trino_client, source_object: str, sql: str):
    return trino_client.query_pandas(sql)

def write_partition_with_replacewhere(
    *,
    pdf,
    target_table: str,
    partition_key: str,
    partition_value: int,
    query_started_at: datetime,
    query_completed_at: datetime,
    allow_empty_overwrite: bool,
) -> int:
    ensure_target_table_exists(target_table)
    if pdf.empty:
        if allow_empty_overwrite:
            spark.sql(f"DELETE FROM {target_table} WHERE {partition_key} = {partition_value}")
        return 0

    spark_df = create_target_dataframe(
        pdf=pdf,
        target_table=target_table,
        query_started_at=query_started_at,
        query_completed_at=query_completed_at,
    )
    (
        spark_df.write.format("delta")
        .mode("overwrite")
        .option("replaceWhere", f"{partition_key} = {partition_value}")
        .saveAsTable(target_table)
    )
    return int(len(pdf.index))

def log_run_summary(*, payload: dict):
    append_log_row(target_table=run_plan["target_tables"]["run_log"], payload=payload)

def log_partition_result(*, payload: dict):
    append_log_row(target_table=run_plan["target_tables"]["partition_log"], payload=payload)


In [ ]:
if dry_run:
    dry_run_summary = {
        "status": "dry_run",
        "run_id": run_id,
        "planned_state_partitions": len(planned_state_queries),
        "planned_flight_partitions": len(planned_flight_queries),
        "target_tables": run_plan["target_tables"],
        "write_policy": {
            "states_partition_key": "hour",
            "flights_partition_key": "day",
            "strategy": "extract_then_replaceWhere",
            "overwrite_empty_partitions": overwrite_empty_partitions,
        },
        "trino_connection": {
            "host": trino_host,
            "port": trino_port,
            "catalog": trino_catalog,
            "schema": trino_schema,
            "credential_mode": "oauth2_external_auth",
        },
    }
    dry_run_summary
else:
    trino_client = build_trino_client()
    try:
        source_runs = [
        {
            "source_name": "opensky_trino",
            "source_object": state_table,
            "target_table": run_plan["target_tables"]["states"],
            "partition_key": "hour",
            "queries": planned_state_queries,
        }
        ]
        if include_flights:
            source_runs.append(
            {
                "source_name": "opensky_trino",
                "source_object": flight_table,
                "target_table": run_plan["target_tables"]["flights"],
                "partition_key": "day",
                "queries": planned_flight_queries,
            }
            )

        run_started_at = utc_now()
        run_summaries = []
        total_failed_partitions = 0
        first_failure_detail = None

        for source_run in source_runs:
            ensure_target_table_exists(source_run["target_table"])
            ensure_target_table_exists(run_plan["target_tables"]["run_log"])
            ensure_target_table_exists(run_plan["target_tables"]["partition_log"])

            success_partition_count = 0
            failed_partition_count = 0
            rows_written_total = 0
            failures = []

            for query_plan in source_run["queries"]:
                partition_started_at = utc_now()
                status = "failed"
                rows_read = 0
                rows_written = 0
                error_message = None
                query_started_at = utc_now()
                query_completed_at = query_started_at

                try:
                    partition_pdf = extract_partition_via_trino(
                    trino_client=trino_client,
                    source_object=source_run["source_object"],
                    sql=query_plan["sql"],
                    )
                    query_completed_at = utc_now()
                    rows_read = int(len(partition_pdf.index))

                    if rows_read == 0:
                        status = "empty_source"
                        rows_written = write_partition_with_replacewhere(
                        pdf=partition_pdf,
                        target_table=source_run["target_table"],
                        partition_key=query_plan["partition_key"],
                        partition_value=query_plan["partition_value"],
                        query_started_at=query_started_at,
                        query_completed_at=query_completed_at,
                        allow_empty_overwrite=overwrite_empty_partitions,
                        )
                    else:
                        status = "success"
                        rows_written = write_partition_with_replacewhere(
                        pdf=partition_pdf,
                        target_table=source_run["target_table"],
                        partition_key=query_plan["partition_key"],
                        partition_value=query_plan["partition_value"],
                        query_started_at=query_started_at,
                        query_completed_at=query_completed_at,
                        allow_empty_overwrite=overwrite_empty_partitions,
                        )

                    success_partition_count += 1
                    rows_written_total += rows_written
                except Exception as exc:
                    query_completed_at = utc_now()
                    failed_partition_count += 1
                    total_failed_partitions += 1
                    error_message = f"{type(exc).__name__}: {exc}"
                    failures.append(
                    {
                        "partition_value": query_plan["partition_value"],
                        "error_message": error_message,
                    }
                    )
                    if first_failure_detail is None:
                        first_failure_detail = {
                        "source_object": source_run["source_object"],
                        "partition_key": query_plan["partition_key"],
                        "partition_value": query_plan["partition_value"],
                        "error_message": error_message,
                        }

                log_partition_result(
                    payload={
                    "run_id": run_id,
                    "pipeline_name": "01a_ingest_opensky_history",
                    "source_object": source_run["source_object"],
                    "target_table": source_run["target_table"],
                    "partition_key": query_plan["partition_key"],
                    "partition_value": str(query_plan["partition_value"]),
                    "status": status,
                    "rows_read": rows_read,
                    "rows_written": rows_written,
                    "attempt_no": 1,
                    "dry_run": False,
                    "started_at": naive_utc(partition_started_at),
                    "completed_at": naive_utc(utc_now()),
                    "error_message": error_message,
                    }
                )

            source_status = "success"
            if failed_partition_count and success_partition_count:
                source_status = "partial_success"
            elif failed_partition_count and not success_partition_count:
                source_status = "failed"

            run_summary = {
                "run_id": run_id,
                "pipeline_name": "01a_ingest_opensky_history",
                "source_name": source_run["source_name"],
                "source_object": source_run["source_object"],
                "target_table": source_run["target_table"],
                "scope_id": scope_id,
                "start_date": start_date,
                "end_date": end_date,
                "partition_key": source_run["partition_key"],
                "planned_partition_count": len(source_run["queries"]),
                "success_partition_count": success_partition_count,
                "failed_partition_count": failed_partition_count,
                "dry_run": False,
                "status": source_status,
                "rows_written": rows_written_total,
                "started_at": naive_utc(run_started_at),
                "completed_at": naive_utc(utc_now()),
                "error_message": failures[0]["error_message"] if failures else None,
                "metadata_json": json_dumps(
                {
                    "catalog": catalog,
                    "focus_airport": focus_airport,
                    "overwrite_empty_partitions": overwrite_empty_partitions,
                    "failed_partitions": failures[:20],
                }
                ),
            }
            log_run_summary(payload=run_summary)
            run_summaries.append(run_summary)

        final_summary = {
            "run_id": run_id,
            "catalog": catalog,
            "status": "success" if total_failed_partitions == 0 else "failed",
            "source_runs": run_summaries,
        }

        if total_failed_partitions:
            raise RuntimeError(
            f"Historical ingestion completed with {total_failed_partitions} failed partitions. "
            f"run_id={run_id}; first_failure={json_dumps(first_failure_detail or {})}"
            )

        final_summary
    finally:
        if hasattr(trino_client, "close"):
            trino_client.close()
